# 01 — Chuẩn hoá dữ liệu (raw CSV → Parquet gọn)

## Vì sao cần bước này?

`transactions_train.csv` nặng 3,5 GB và lưu mọi thứ dạng chuỗi. Nếu đọc lại ở mỗi bước, mỗi lần
lặp thực nghiệm sẽ mất hàng chục giây và rất tốn RAM. Ta chuẩn hoá **một lần**:

| Vấn đề | Cách xử lý |
|---|---|
| `customer_id` là chuỗi 64 ký tự, lặp 31,8M lần | Ánh xạ sang `cust_int` (Int32) |
| `article_id` là chuỗi 10 ký tự | Ánh xạ sang `art_int` (Int32) |
| `t_dat` là chuỗi `YYYY-MM-DD` | Đổi sang `day_idx` = số ngày kể từ 2018-09-20 (Int16) |
| `price` float64 | `Float32` (đủ chính xác, giảm một nửa bộ nhớ) |
| Cột categorical của `articles` | Mã hoá top-K phổ biến + nhóm `OTHER` |
| Toàn bộ file 3,5 GB | Ghi Parquet nén zstd (~185 MB) |

Kết quả (trong `artifacts/`):

| File | Nội dung |
|---|---|
| `customers_base.parquet` | khách hàng + `cust_int` + `active_flag` |
| `articles_encoded.parquet` | `art_int` + mã category top-K |
| `transactions_compact.parquet` | `cust_int`, `art_int`, `day_idx`, `price`, `channel` |

> Logic nằm trong `src/prepare_data.py` để notebook và pipeline dùng chung một nguồn.

In [1]:
import sys
from pathlib import Path

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

# Notebook có thể được mở từ thư mục notebooks/ hoặc từ gốc repo
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

DATA = ROOT / "data"
CUSTOMERS = DATA / "customers.csv"
ARTICLES = DATA / "articles.csv"
TXN = DATA / "transactions_train.csv"
ARTIFACTS = ROOT / "artifacts"
OUTPUTS = ROOT / "outputs"
REPORTS = ROOT / "reports"

plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.grid": True, "grid.alpha": 0.3})
print("polars", pl.__version__, "| numpy", np.__version__, "| ROOT =", ROOT)

polars 2.1.0 | numpy 2.3.5 | ROOT = E:\download\age_imputation


In [2]:
import json

from src.prepare_data import prepare_articles, prepare_customers, prepare_transactions
import src.config as C

C.ARTIFACTS.mkdir(parents=True, exist_ok=True)

customers = prepare_customers()
customers.write_parquet(C.ARTIFACTS / "customers_base.parquet")
print(f"customers_base: {customers.shape}")
customers.head(3)

customers_base: (1371980, 9)


customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code,cust_int,active_flag
str,f32,i8,str,str,f32,str,i32,i8
"""00000dbacae5abe5e23885899a1fa4…",null,0,"""ACTIVE""","""NONE""",49.0,"""52043ee2162cf5aa7ee79974281641…",0,0
"""0000423b00ade91418cceaf3b26c6a…",null,0,"""ACTIVE""","""NONE""",25.0,"""2973abc54daa8a5f8ccfe9362140c6…",1,0
"""000058a12d5b43e67d225668fa1f8d…",null,0,"""ACTIVE""","""NONE""",24.0,"""64f17e6a330a85798e4998f62d0930…",2,0


### Mã hoá `articles`

Với mỗi cột categorical (`index_group_name`, `index_name`, `garment_group_name`, `section_name`,
`product_type_name`) ta giữ **K giá trị phổ biến nhất** và gộp phần còn lại vào `OTHER`.

Lý do: giá trị hiếm (một vài sản phẩm) không đủ dữ liệu để ước lượng tuổi trung bình — giữ lại chỉ
tạo nhiễu và làm phình số chiều. Ở notebook 03 ta dùng thêm bản **đầy đủ** (`articles_levels.parquet`)
cho target encoding, nơi làm trơn Bayes xử lý được cả mức hiếm.

In [3]:
articles_enc, maps = prepare_articles()
articles_enc.write_parquet(C.ARTIFACTS / "articles_encoded.parquet")
(C.ARTIFACTS / "category_maps.json").write_text(json.dumps(maps, ensure_ascii=False, indent=2))

print(f"articles_encoded: {articles_enc.shape}")
print("\nSố category giữ lại cho từng cột:")
for col, values in maps.items():
    print(f"  {col:<20} {len(values) - 1} + OTHER   ví dụ: {values[:2]}")

articles_enc.head(3)

articles_encoded: (105542, 7)

Số category giữ lại cho từng cột:
  index_group_name     5 + OTHER   ví dụ: ['Ladieswear', 'Baby/Children']
  index_name           10 + OTHER   ví dụ: ['Ladieswear', 'Divided']
  garment_group_name   15 + OTHER   ví dụ: ['Jersey Fancy', 'Accessories']
  section_name         15 + OTHER   ví dụ: ['Womens Everyday Collection', 'Divided Collection']
  product_type_name    20 + OTHER   ví dụ: ['Trousers', 'Dress']


article_id,art_int,c_index_group_name,c_index_name,c_garment_group_name,c_section_name,c_product_type_name
str,i32,i16,i16,i16,i16,i16
"""0108775015""",0,0,0,2,15,9
"""0108775044""",1,0,0,2,15,9
"""0108775051""",2,0,0,2,15,9


### Giao dịch dạng số nguyên

Bước nặng nhất: join 31,8M dòng với bảng khách và bảng sản phẩm rồi ghi Parquet.
Dùng **streaming engine** của Polars để bộ nhớ không tăng theo kích thước file.

In [4]:
prepare_transactions(customers, articles_enc)

compact = pl.scan_parquet(C.TXN_PARQUET)
print(
    compact.select(
        n=pl.len(),
        n_customers=pl.col("cust_int").n_unique(),
        n_articles=pl.col("art_int").n_unique(),
        day_min=pl.col("day_idx").min(),
        day_max=pl.col("day_idx").max(),
    ).collect(engine="streaming")
)
print(f"dung lượng: {C.TXN_PARQUET.stat().st_size / 1e6:.0f} MB (từ 3.5 GB CSV)")

shape: (1, 5)
┌──────────┬─────────────┬────────────┬─────────┬─────────┐
│ n        ┆ n_customers ┆ n_articles ┆ day_min ┆ day_max │
│ ---      ┆ ---         ┆ ---        ┆ ---     ┆ ---     │
│ u32      ┆ u32         ┆ u32        ┆ i16     ┆ i16     │
╞══════════╪═════════════╪════════════╪═════════╪═════════╡
│ 31788324 ┆ 1362281     ┆ 104547     ┆ 0       ┆ 733     │
└──────────┴─────────────┴────────────┴─────────┴─────────┘
dung lượng: 194 MB (từ 3.5 GB CSV)


In [5]:
compact.head(5).collect()

cust_int,art_int,day_idx,price,channel
i32,i32,i16,f32,i8
9205,35031,0,0.044475,2
9530,464,0,0.033881,1
9177,27180,0,0.022017,2
10761,21611,0,0.011847,1
11990,28044,0,0.008458,2


## Kiểm tra tính đúng đắn

Kiểm tra nhanh nhưng quan trọng: **không được mất dòng nào**.

- Số giao dịch trong Parquet phải bằng số dòng CSV gốc: **31.788.324**.
- `cust_int` nằm trong `[0, số khách)`; `art_int` trong `[0, số sản phẩm)`.
- `day_idx` trong `[0, 733]` (2018-09-20 → 2020-09-22).

In [6]:
with open(TXN, "r", encoding="utf-8") as f:
    raw_rows = sum(1 for _ in f) - 1

stats = compact.select(
    n=pl.len(),
    cust_max=pl.col("cust_int").max(),
    art_max=pl.col("art_int").max(),
    day_min=pl.col("day_idx").min(),
    day_max=pl.col("day_idx").max(),
    nulls=pl.col("cust_int").is_null().sum() + pl.col("art_int").is_null().sum(),
).collect()

print(f"CSV gốc     : {raw_rows:,} dòng")
print(f"Parquet gọn : {stats['n'][0]:,} dòng")
assert stats["n"][0] == raw_rows, "Mất dòng khi chuyển đổi!"
assert stats["nulls"][0] == 0, "Có join thất bại (null key)!"
assert stats["cust_max"][0] < customers.height
assert stats["art_max"][0] < articles_enc.height
print("✅ Số dòng khớp, không có null, chỉ số nằm trong miền hợp lệ")
stats

CSV gốc     : 31,788,324 dòng
Parquet gọn : 31,788,324 dòng
✅ Số dòng khớp, không có null, chỉ số nằm trong miền hợp lệ


n,cust_max,art_max,day_min,day_max,nulls
u32,i32,i32,i16,i16,u32
31788324,1371979,105539,0,733,0


## Tóm tắt

| Artifact | Vai trò |
|---|---|
| `artifacts/customers_base.parquet` | nguồn khách hàng + nhãn `age` |
| `artifacts/articles_encoded.parquet` | category top-K để dựng tỷ trọng mua |
| `artifacts/category_maps.json` | tên category ↔ mã (giải thích feature) |
| `artifacts/transactions_compact.parquet` | 31,8M giao dịch, ~185 MB, đọc trong vài giây |

➡️ Tiếp theo: **02_features.ipynb** — dựng đặc trưng từ các artifact này.